# D.R.O.N.A. — Exploratory Data Analysis

Explore the ChromaDB knowledge base contents:
- Document counts and tier breakdown (Nepal vs International)
- Source type distribution
- Embedding space visualisation (UMAP)
- Job skill frequency (Nepal postings)
- Curriculum module coverage

**Prerequisites:** Run `python scripts/ingest_data.py` first to populate ChromaDB.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import Counter

# Suppress noisy warnings
import warnings
warnings.filterwarnings('ignore')

from drona.utils.settings import settings
from drona.utils.logging import setup_logging
setup_logging('WARNING')

print('Settings loaded. ChromaDB path:', settings.chromadb_dir)

## 1. Load ChromaDB Collections

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(settings.chromadb_dir))
collections = client.list_collections()
print(f'Collections: {[c.name for c in collections]}')

all_docs = []
for col in collections:
    result = col.get(include=['documents', 'metadatas'])
    for doc, meta, doc_id in zip(result['documents'], result['metadatas'], result['ids']):
        all_docs.append({
            'id': doc_id,
            'collection': col.name,
            'text': doc,
            **meta,
        })

df = pd.DataFrame(all_docs)
print(f'\nTotal documents: {len(df)}')
df.head()

## 2. Collection and Tier Breakdown

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Collection counts
col_counts = df['collection'].value_counts()
axes[0].bar(col_counts.index, col_counts.values, color=['#2196F3', '#FF9800'])
axes[0].set_title('Documents per Collection')
axes[0].set_ylabel('Count')
for i, v in enumerate(col_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Tier breakdown
if 'tier' in df.columns:
    tier_counts = df['tier'].value_counts()
    colours = {'nepal': '#4CAF50', 'international': '#2196F3', 'synthetic': '#FF9800'}
    bar_colours = [colours.get(t, '#9E9E9E') for t in tier_counts.index]
    axes[1].bar(tier_counts.index, tier_counts.values, color=bar_colours)
    axes[1].set_title('Documents by Data Tier')
    axes[1].set_ylabel('Count')
    for i, v in enumerate(tier_counts.values):
        axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Source type breakdown
if 'source_type' in df.columns:
    src_counts = df['source_type'].value_counts()
    axes[2].pie(src_counts.values, labels=src_counts.index, autopct='%1.1f%%',
                colors=cm.Set3(np.linspace(0, 1, len(src_counts))))
    axes[2].set_title('Source Type Distribution')

plt.tight_layout()
plt.savefig('../data/evaluation/eda_collection_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to data/evaluation/eda_collection_breakdown.png')

## 3. Document Length Distribution

In [ ]:
df['text_len'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['text_len'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Document length (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Document Length Distribution')
axes[0].axvline(df['text_len'].median(), color='red', linestyle='--', label=f"Median={df['text_len'].median():.0f}")
axes[0].legend()

if 'tier' in df.columns:
    for tier, grp in df.groupby('tier'):
        axes[1].hist(grp['text_len'], bins=30, alpha=0.6, label=tier)
    axes[1].set_xlabel('Document length (chars)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Length by Tier')
    axes[1].legend()

plt.tight_layout()
plt.show()

print(df['text_len'].describe().round(0))

## 4. Nepal Job Market — Top Skills

In [ ]:
import re

# Extract skill keywords from Nepal job posting texts
SKILL_KEYWORDS = [
    'python', 'javascript', 'react', 'node.js', 'django', 'flask',
    'java', 'spring', 'sql', 'postgresql', 'mysql', 'mongodb',
    'aws', 'docker', 'kubernetes', 'git', 'linux', 'rest api',
    'machine learning', 'deep learning', 'data science', 'tensorflow',
    'pytorch', 'pandas', 'numpy', 'tableau', 'power bi',
    'php', 'laravel', '.net', 'c#', 'c++', 'go', 'rust',
    'flutter', 'android', 'ios', 'react native',
    'devops', 'ci/cd', 'agile', 'scrum', 'jira',
    'cybersecurity', 'networking', 'cloud computing',
]

nepal_jobs = df[(df.get('tier', pd.Series()) == 'nepal') & 
                (df.get('source_type', pd.Series()) == 'job_posting')] if 'tier' in df.columns else df

skill_counts = Counter()
for text in nepal_jobs['text'].str.lower():
    for skill in SKILL_KEYWORDS:
        if skill in text:
            skill_counts[skill] += 1

if skill_counts:
    top_skills = pd.DataFrame(skill_counts.most_common(20), columns=['skill', 'count'])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top_skills['skill'][::-1], top_skills['count'][::-1], color='steelblue')
    ax.set_xlabel('Frequency in Nepal Job Postings')
    ax.set_title('Top 20 Skills Demanded — Nepal IT Job Market')
    for bar, count in zip(bars, top_skills['count'][::-1]):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                str(count), va='center')
    plt.tight_layout()
    plt.savefig('../data/evaluation/eda_nepal_skills.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(top_skills.to_string(index=False))
else:
    print('No Nepal job postings found. Run scripts/ingest_data.py first.')

## 5. Embedding Space Visualisation (UMAP)

In [ ]:
try:
    import umap
    HAS_UMAP = True
except ImportError:
    print('umap-learn not installed. Install with: pip install umap-learn')
    print('Showing PCA fallback instead.')
    HAS_UMAP = False

from sklearn.decomposition import PCA

# Retrieve embeddings from ChromaDB
all_embeddings = []
all_meta = []
for col in collections:
    result = col.get(include=['embeddings', 'metadatas'])
    if result['embeddings']:
        all_embeddings.extend(result['embeddings'])
        all_meta.extend(result['metadatas'])

if all_embeddings:
    X = np.array(all_embeddings)
    print(f'Embedding matrix: {X.shape}')
    
    if HAS_UMAP and X.shape[0] >= 10:
        reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=min(15, X.shape[0]-1))
        coords = reducer.fit_transform(X)
        method = 'UMAP'
    else:
        coords = PCA(n_components=2, random_state=42).fit_transform(X)
        method = 'PCA'
    
    tiers = [m.get('tier', 'unknown') for m in all_meta]
    tier_colours = {'nepal': '#4CAF50', 'international': '#2196F3', 'synthetic': '#FF9800', 'unknown': '#9E9E9E'}
    
    fig, ax = plt.subplots(figsize=(10, 8))
    for tier in set(tiers):
        mask = [t == tier for t in tiers]
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=tier_colours.get(tier, '#9E9E9E'), label=tier, alpha=0.6, s=20)
    ax.set_title(f'Document Embedding Space ({method})')
    ax.set_xlabel(f'{method}-1')
    ax.set_ylabel(f'{method}-2')
    ax.legend(markerscale=2)
    plt.tight_layout()
    plt.savefig(f'../data/evaluation/eda_embeddings_{method.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No embeddings found. Ensure ChromaDB is populated with dense embeddings.')

## 6. Nepal Fraction Summary

Key metric for Contribution C4: ≥ 60% of documents should be Nepal-tier.

In [ ]:
if 'tier' in df.columns:
    nepal_count = (df['tier'] == 'nepal').sum()
    total = len(df)
    nepal_frac = nepal_count / total if total > 0 else 0
    print(f'Total documents:  {total}')
    print(f'Nepal-tier docs:  {nepal_count} ({nepal_frac:.1%})')
    print(f'C4 target (≥60%): {"✅ PASS" if nepal_frac >= 0.6 else "❌ FAIL — add more Nepal data"}')
    print()
    print(df.groupby(['tier', 'source_type']).size().to_string())
else:
    print('No tier metadata found. Ensure ChromaDB is populated.')